# 02 — CV, validation và test

Bắt đầu một pipeline Original + NB; sau đó xem bảng 28 cấu hình. Quy tắc chọn dùng CV mean, hòa điểm dùng validation F1, rồi tên model/variant.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experiment.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from heart_disease.config import load_settings
from heart_disease.data import load_cleveland, split_dataset
settings = load_settings()
raw = load_cleveland(settings.raw_path)
splits = split_dataset(raw, settings)


In [ ]:
from heart_disease.models import make_estimator, parameter_grid
from heart_disease.pipelines import build_model_pipeline
from heart_disease.training import tune_pipeline
from heart_disease.evaluation import classification_metrics, disease_probability

pipe = build_model_pipeline("original", make_estimator("naive_bayes", settings.seed), settings)
search = tune_pipeline(pipe, parameter_grid("naive_bayes"), *splits["train"], settings)
X_val, y_val = splits["val"]
print(search.best_params_, search.best_score_)
print(classification_metrics(y_val, search.predict(X_val), disease_probability(search, X_val)))


## Bảng validation và lựa chọn đã khóa

Nếu đã có benchmark, đọc lại kết quả. Muốn chủ động chạy lại, dùng CLI benchmark. Không chọn lại theo điểm test.

In [ ]:
import json
import pandas as pd
from heart_disease.workflows import benchmark, evaluate
manifest_path = ROOT / "artifacts/metadata/benchmark.json"
if not manifest_path.exists():
    benchmark(settings)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print("Selected before test:", manifest["selected_key"])
display(pd.read_csv(ROOT / "reports/validation_results.csv"))


## Báo cáo test và độ ổn định

Evaluate dùng lại báo cáo nếu run đã được đánh giá. Repeated CV chỉ dùng TRAIN; mô tả độ ổn định của cấu hình đã chọn, không thay nested CV.

In [ ]:
evaluation = evaluate(settings)
display(pd.read_csv(ROOT / "reports/results.csv"))
print(evaluation["stability_training_only"])


## Thí nghiệm bổ sung (hậu nghiệm)
KMeans/stacking thêm sau khi đã xem test chính. Chỉ đối chiếu mô tả, không chọn lại demo.

In [ ]:
from heart_disease.extension_workflow import run_extensions
extension_manifest = run_extensions(settings)
display(pd.read_csv(ROOT / "reports/extensions" / manifest["run_id"] / "results.csv"))


## Phân tích benchmark đã hoàn thiện

RF Original được chọn trước test bằng CV accuracy 0.8263 ± 0.0181. Test có TN=15, FP=2, FN=1, TP=13; accuracy 0.9032, recall lớp 1 0.9286. Một false negative ở dòng CSV 67 có xác suất 0,3862; không đổi ngưỡng theo lỗi này. FE chỉ tăng CV ở 2/7 mô hình, Original+DT ở 2/7 nên không có cải thiện nhất quán. Một mẫu test làm accuracy đổi 3,23 điểm phần trăm. Repeated CV F1 trên train 0.7855 ± 0.0589 là độ ổn định mô tả, không thay nested CV. KMeans/stacking nằm trong bảng riêng, thêm sau khi đã xem test, không dùng để chọn lại demo.